# 10 Noise, Latency, Gain Stability

This notebook fills the main engineering gap left by the clean high-order AO demonstrator: how measurement noise, loop delay, and controller gain trade against residual wavefront error and science-band Strehl.

The model is intentionally compact. It uses the existing geometric SH-WFS / Gaussian-DM control loop, then adds an explicit centroid-noise proxy instead of a full detector propagation at every frame. The results should be read as a reproducible control-trade study, not as a calibrated telescope prediction.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from ao_closed_loop import (
    actuator_centers_on_pupil,
    build_dm_wfs_response_matrix,
    gaussian_influence_functions,
    reconstruct_dm_delta,
    shifted_atmosphere,
    synthesize_dm_phase,
)
from phase_screen import fourier_phase_screen, rms
from reconstruction import measure_slopes

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

## Compact AO Plant

The simulation uses a modest pupil grid so the scans run quickly on a laptop. The wavefront is expressed in radians at the WFS wavelength; RMS values are converted to OPD and H-band Strehl for science-facing interpretation.

In [ ]:
WFS_WAVELENGTH_M = 700e-9
SCI_WAVELENGTH_M = 1.65e-6

N = 96
DIAMETER_M = 1.0
DELTA_M = DIAMETER_M / N
N_LENSLETS = 10
N_ACTUATORS = 9
N_STEPS = 55
SETTLING = 20

phase0, X, Y, pupil_mask = fourier_phase_screen(
    N=N,
    delta=DELTA_M,
    r0=0.16,
    diameter=DIAMETER_M,
    wavelength=WFS_WAVELENGTH_M,
    seed=11,
    target_rms_rad=4.0,
    normalize_rms=True,
)

actuator_centers, pitch = actuator_centers_on_pupil(DIAMETER_M, N_ACTUATORS)
influence = gaussian_influence_functions(X, Y, pupil_mask, actuator_centers, pitch, coupling=0.36)
dm_response, wfs_centers = build_dm_wfs_response_matrix(
    influence,
    pupil_mask,
    X,
    Y,
    n_lenslets=N_LENSLETS,
    min_fill=0.35,
)

print(f"DM actuators inside pupil: {influence.shape[0]}")
print(f"Valid SH-WFS subapertures: {len(wfs_centers)}")
print(f"Open-loop phase RMS: {rms(phase0, pupil_mask):.2f} rad at {WFS_WAVELENGTH_M * 1e9:.0f} nm")

In [ ]:
def opd_nm_from_wfs_phase_rms(phase_rms_rad):
    return phase_rms_rad * WFS_WAVELENGTH_M / (2.0 * np.pi) * 1e9


def marechal_strehl_from_wfs_phase_rms(phase_rms_rad):
    science_phase_rms = phase_rms_rad * WFS_WAVELENGTH_M / SCI_WAVELENGTH_M
    return float(np.exp(-(science_phase_rms**2)))


def centroid_noise_to_slope_std(photons, read_noise_e=0.0, reference_std=0.018):
    """Simple centroid-noise proxy mapped into geometric slope units."""
    if photons is None or np.isinf(photons):
        return 0.0
    photons = max(float(photons), 1.0)
    photon_term = np.sqrt(1.0e4 / photons)
    read_term = (read_noise_e / 5.0) * (1.0e4 / photons)
    return float(reference_std * np.sqrt(photon_term**2 + read_term**2))


def run_loop(
    gain=0.45,
    delay_frames=1,
    photons=1.0e4,
    read_noise_e=2.0,
    n_steps=N_STEPS,
    rcond=2e-2,
    command_leak=0.01,
    seed=1,
):
    rng = np.random.default_rng(seed)
    commands = np.zeros(influence.shape[0], dtype=float)
    measurement_buffer = []
    noise_std = centroid_noise_to_slope_std(photons, read_noise_e=read_noise_e)

    history = {"rms_before": [], "rms_after": [], "command_norm": []}
    for k in range(n_steps):
        atm = shifted_atmosphere(phase0, pupil_mask, shift_x_pix=k, shift_y_pix=0)
        dm_before = synthesize_dm_phase(commands, influence, pupil_mask)
        residual_before = np.where(pupil_mask, atm - dm_before, np.nan)
        residual_before[pupil_mask] -= np.nanmean(residual_before[pupil_mask])

        _, slopes = measure_slopes(
            residual_before,
            pupil_mask,
            X,
            Y,
            n_lenslets=N_LENSLETS,
            min_fill=0.35,
        )
        noisy_slopes = slopes + rng.normal(scale=noise_std, size=slopes.shape)
        measurement_buffer.append(noisy_slopes)

        if len(measurement_buffer) > delay_frames:
            control_slopes = measurement_buffer.pop(0)
        else:
            control_slopes = np.zeros_like(slopes)

        delta, *_ = reconstruct_dm_delta(control_slopes, dm_response, rcond=rcond)
        commands = (1.0 - command_leak) * commands + gain * delta

        dm_after = synthesize_dm_phase(commands, influence, pupil_mask)
        residual_after = np.where(pupil_mask, atm - dm_after, np.nan)
        residual_after[pupil_mask] -= np.nanmean(residual_after[pupil_mask])

        history["rms_before"].append(rms(residual_before, pupil_mask))
        history["rms_after"].append(rms(residual_after, pupil_mask))
        history["command_norm"].append(float(np.linalg.norm(commands)))

    return {key: np.asarray(value) for key, value in history.items()} | {"noise_std": noise_std}


def summarize_run(label, history):
    tail = history["rms_after"][SETTLING:]
    median_rms = float(np.nanmedian(tail))
    return {
        "case": label,
        "median_residual_rms_rad": median_rms,
        "median_residual_opd_nm": opd_nm_from_wfs_phase_rms(median_rms),
        "h_band_strehl": marechal_strehl_from_wfs_phase_rms(median_rms),
        "final_command_norm": float(history["command_norm"][-1]),
        "noise_std": float(history["noise_std"]),
    }

## Photon Flux And Read Noise

The centroid-noise proxy is intentionally transparent: photon noise scales roughly as `1/sqrt(N_photon)`, while read noise becomes important faster at low flux. The read-noise panel below is therefore run as a low-flux stress case and averaged over a few random seeds, so the plotted trend is not dominated by one noise realization.

In [ ]:
photon_levels = [2e2, 5e2, 1e3, 3e3, 1e4, 3e4, np.inf]
photon_rows = []
for photons in photon_levels:
    hist = run_loop(gain=0.45, delay_frames=1, photons=photons, read_noise_e=2.0, seed=12)
    label = "infinite" if np.isinf(photons) else f"{photons:.0f}"
    row = summarize_run(label, hist)
    row["photons_per_subaperture"] = photons
    photon_rows.append(row)

photon_scan = pd.DataFrame(photon_rows)
photon_scan

In [ ]:
read_noise_levels = [0.0, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]
read_noise_photons = 1.0e2
read_noise_seeds = [14, 15, 16]
read_noise_rows = []
for rn in read_noise_levels:
    trial_rows = []
    for seed in read_noise_seeds:
        hist = run_loop(gain=0.45, delay_frames=1, photons=read_noise_photons, read_noise_e=rn, seed=seed)
        trial_rows.append(summarize_run(f"read_noise_{rn:g}e", hist))
    row = {
        "case": f"read_noise_{rn:g}e",
        "median_residual_rms_rad": float(np.mean([r["median_residual_rms_rad"] for r in trial_rows])),
        "median_residual_opd_nm": float(np.mean([r["median_residual_opd_nm"] for r in trial_rows])),
        "std_residual_opd_nm": float(np.std([r["median_residual_opd_nm"] for r in trial_rows])),
        "h_band_strehl": float(np.mean([r["h_band_strehl"] for r in trial_rows])),
        "final_command_norm": float(np.mean([r["final_command_norm"] for r in trial_rows])),
        "noise_std": float(np.mean([r["noise_std"] for r in trial_rows])),
    }
    row["read_noise_e"] = rn
    row["photons_per_subaperture"] = read_noise_photons
    row["n_trials"] = len(read_noise_seeds)
    read_noise_rows.append(row)

read_noise_scan = pd.DataFrame(read_noise_rows)
read_noise_scan

## Gain, Delay, And Stability Boundary

Frame delay shifts the controller toward instability because commands are based on older residuals. The scan below treats delay as both frames and physical latency for a nominal 1 kHz loop. A cell is marked unstable if any seed has non-finite residuals, residuals worse than open loop, or excessive command growth.

In [ ]:
gains = [0.15, 0.30, 0.45, 0.60, 0.75, 0.90]
delays = [0, 1, 2, 3]
gain_delay_seeds = [21, 22, 23]
nominal_frame_rate_hz = 1000.0
command_norm_limit = 200.0
open_loop_rms = rms(phase0, pupil_mask)
open_loop_opd_nm = opd_nm_from_wfs_phase_rms(open_loop_rms)

gain_delay_rows = []
for delay in delays:
    for gain in gains:
        trial_rows = []
        trial_stable = []
        tail_command_norms = []
        for seed in gain_delay_seeds:
            hist = run_loop(gain=gain, delay_frames=delay, photons=3.0e3, read_noise_e=2.0, seed=seed + 10 * delay)
            row_trial = summarize_run(f"gain_{gain:.2f}_delay_{delay}", hist)
            tail = hist["rms_after"][SETTLING:]
            tail_commands = hist["command_norm"][SETTLING:]
            is_stable = bool(
                np.all(np.isfinite(hist["rms_after"]))
                and np.nanmedian(tail) < open_loop_rms
                and hist["command_norm"][-1] < command_norm_limit
                and np.nanmax(tail_commands) < command_norm_limit
            )
            trial_rows.append(row_trial)
            trial_stable.append(is_stable)
            tail_command_norms.append(float(np.nanmedian(tail_commands)))
        row = {
            "case": f"gain_{gain:.2f}_delay_{delay}",
            "median_residual_rms_rad": float(np.median([r["median_residual_rms_rad"] for r in trial_rows])),
            "median_residual_opd_nm": float(np.median([r["median_residual_opd_nm"] for r in trial_rows])),
            "std_residual_opd_nm": float(np.std([r["median_residual_opd_nm"] for r in trial_rows])),
            "h_band_strehl": float(np.median([r["h_band_strehl"] for r in trial_rows])),
            "final_command_norm": float(np.median([r["final_command_norm"] for r in trial_rows])),
            "median_tail_command_norm": float(np.median(tail_command_norms)),
            "noise_std": float(np.median([r["noise_std"] for r in trial_rows])),
            "gain": gain,
            "delay_frames": delay,
            "latency_ms_at_1khz": float(delay / nominal_frame_rate_hz * 1e3),
            "stable_fraction": float(np.mean(trial_stable)),
            "stable": bool(np.all(trial_stable)),
            "n_trials": len(gain_delay_seeds),
            "open_loop_opd_nm": open_loop_opd_nm,
        }
        gain_delay_rows.append(row)

gain_delay_scan = pd.DataFrame(gain_delay_rows)
gain_delay_scan.pivot(index="delay_frames", columns="gain", values="median_residual_opd_nm")

In [ ]:
output_dir = ROOT / "figures"
output_dir.mkdir(exist_ok=True)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)

finite_photons = photon_scan[np.isfinite(photon_scan["photons_per_subaperture"])]
axes[0].semilogx(
    finite_photons["photons_per_subaperture"],
    finite_photons["median_residual_opd_nm"],
    marker="o",
)
axes[0].set_xlabel("photons per subaperture per frame")
axes[0].set_ylabel("median residual OPD RMS [nm]")
axes[0].set_title("Photon-flux sensitivity")

axes[1].errorbar(
    read_noise_scan["read_noise_e"],
    read_noise_scan["median_residual_opd_nm"],
    yerr=read_noise_scan["std_residual_opd_nm"],
    marker="s",
    color="#0b7a75",
    capsize=3,
)
axes[1].set_xlabel("read noise [e- RMS]")
axes[1].set_ylabel("median residual OPD RMS [nm]")
axes[1].set_title("Read-noise sensitivity")

heatmap = gain_delay_scan.pivot(index="delay_frames", columns="gain", values="median_residual_opd_nm")
stable_map = gain_delay_scan.pivot(index="delay_frames", columns="gain", values="stable").astype(bool)
plot_values = heatmap.clip(upper=1000.0)
im = axes[2].imshow(plot_values.values, origin="lower", aspect="auto", cmap="viridis")

contour_levels = [220.0, 300.0, 500.0, 1000.0]
valid_levels = [level for level in contour_levels if np.nanmin(heatmap.values) < level < np.nanmax(heatmap.values)]
if valid_levels:
    contours = axes[2].contour(heatmap.values, levels=valid_levels, colors="white", linewidths=0.8)
    axes[2].clabel(contours, inline=True, fontsize=7, fmt="%.0f nm")

unstable_cells = np.argwhere(~stable_map.values)
for iy, ix in unstable_cells:
    axes[2].add_patch(
        Rectangle(
            (ix - 0.5, iy - 0.5),
            1.0,
            1.0,
            facecolor="none",
            edgecolor="white",
            hatch="////",
            linewidth=0.0,
        )
    )

selected_gain = 0.45
selected_delay = 1
selected_x = list(heatmap.columns).index(selected_gain)
selected_y = list(heatmap.index).index(selected_delay)
axes[2].scatter(
    [selected_x],
    [selected_y],
    marker="*",
    s=260,
    facecolor="#ff4d00",
    edgecolor="black",
    linewidth=1.2,
    zorder=6,
)
axes[2].annotate(
    "chosen",
    xy=(selected_x, selected_y),
    xytext=(selected_x + 0.55, selected_y - 0.45),
    color="white",
    fontsize=7,
    arrowprops={"arrowstyle": "->", "color": "#ff4d00", "linewidth": 1.2},
    bbox={"boxstyle": "round,pad=0.15", "facecolor": "black", "edgecolor": "none", "alpha": 0.65},
    zorder=7,
)
axes[2].set_xticks(range(len(heatmap.columns)), [f"{g:.2f}" for g in heatmap.columns])
axes[2].set_yticks(
    range(len(heatmap.index)),
    [f"{d}\n{d / nominal_frame_rate_hz * 1e3:.0f} ms" for d in heatmap.index],
)
axes[2].set_xlabel("loop gain")
axes[2].set_ylabel("delay [frames] / latency at 1 kHz")
axes[2].set_title("Gain-delay stability map")
axes[2].legend(
    handles=[
        Line2D([0], [0], marker="*", color="none", markerfacecolor="#ff4d00", markeredgecolor="black", markersize=11, label="chosen point"),
        Patch(facecolor="none", edgecolor="white", hatch="////", label="unstable / command-limited"),
    ],
    loc="upper left",
    fontsize=7,
    framealpha=0.85,
)
fig.colorbar(im, ax=axes[2], label="OPD RMS [nm], clipped at 1000", fraction=0.046)

figure_path = output_dir / "noise_latency_gain_stability.png"
fig.savefig(figure_path, dpi=140, pil_kwargs={"optimize": True})
figure_path

In [ ]:
generated_dir = ROOT / "figures" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)
photon_scan.to_csv(generated_dir / "noise_latency_photon_scan.csv", index=False)
read_noise_scan.to_csv(generated_dir / "noise_latency_read_noise_scan.csv", index=False)
gain_delay_scan.to_csv(generated_dir / "noise_latency_gain_delay_scan.csv", index=False)

best = gain_delay_scan[gain_delay_scan["stable"]].sort_values("median_residual_opd_nm").head(5)
best[["gain", "delay_frames", "latency_ms_at_1khz", "median_residual_opd_nm", "median_tail_command_norm", "stable_fraction"]]

## Interpretation

* Photon flux mainly controls the floor through centroid noise. Once the proxy noise is small, the residual is dominated by fitting error, delay, and the simplified DM/WFS geometry.
* Read noise matters most in the low-flux regime because it enters before centroid estimation.
* Higher gain is not automatically better. With one or more delayed frames, aggressive gain can raise the tail residual or drive command growth. The gain-delay map therefore marks unstable or command-limited cells instead of treating every low-color cell as an acceptable operating point.
* This notebook complements the clean high-order PSF notebook. It does not replace a full AO error budget with calibrated guide-star magnitude, throughput, WFS wavelength, sampling, vibration, multi-layer turbulence, aliasing, and servo-lag terms.